# The Adjoint Method for Neural ODEs

This notebook walks through the derivation of the adjoint equation used to efficiently compute gradients through ODE solvers — the core idea behind Neural ODEs (Chen et al., 2018).

We build from a simple scalar example to the full general algorithm.

---
## Contents
1. [Motivation: from ResNets to Neural ODEs](#motivation)
2. [Setup: a simple scalar ODE](#setup)
3. [The naive gradient approach](#naive)
4. [Deriving the adjoint equation](#adjoint)
5. [Visualising the adjoint on our example](#visualise)
6. [The full adjoint algorithm](#algorithm)
7. [Numerical demo: gradient check](#demo)
8. [Why memory cost is O(1)](#memory)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.integrate import solve_ivp

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f8f8',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 12,
})

---
<a id='motivation'></a>
## 1. Motivation: from ResNets to Neural ODEs

A standard **residual network** applies a chain of discrete updates:

$$h_{t+1} = h_t + f_\theta(h_t, t)$$

If we shrink the step size and take the limit as the number of layers $\to \infty$, this becomes a **continuous-time ODE**:

$$\frac{dh}{dt} = f_\theta(h(t), t), \qquad h(0) = x$$

The output is $h(T)$ for some terminal time $T$. A black-box ODE solver replaces the discrete forward pass.

**The problem:** to train $\theta$ with gradient descent, we need $\frac{dL}{d\theta}$. How do we backpropagate *through an ODE solver*?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: discrete ResNet layers
ax = axes[0]
ax.set_xlim(-0.5, 5.5)
ax.set_ylim(-0.5, 3)
ax.set_title('ResNet: discrete layers', fontsize=13, fontweight='bold')
ax.axis('off')

colors = plt.cm.Blues(np.linspace(0.3, 0.85, 5))
for i in range(5):
    rect = mpatches.FancyBboxPatch((i*1.05, 0.8), 0.85, 1.2,
                                    boxstyle='round,pad=0.05',
                                    facecolor=colors[i], edgecolor='steelblue', linewidth=1.2)
    ax.add_patch(rect)
    ax.text(i*1.05 + 0.425, 1.4, f'$h_{i}$', ha='center', va='center',
            fontsize=13, fontweight='bold', color='white')
    if i < 4:
        ax.annotate('', xy=(i*1.05+1.0, 1.4), xytext=(i*1.05+0.85, 1.4),
                    arrowprops=dict(arrowstyle='->', color='#333', lw=1.5))
        ax.text(i*1.05 + 0.925, 1.65, f'$+f_\\theta$', ha='center', fontsize=9, color='#555')

ax.text(2.6, 0.2, 'Finite layers — backprop stores all $h_t$', ha='center',
        fontsize=10, color='#555', style='italic')

# Right: Neural ODE continuous trajectory
ax2 = axes[1]
ax2.set_title('Neural ODE: continuous trajectory', fontsize=13, fontweight='bold')

t = np.linspace(0, 1, 300)
theta = 1.5
h = np.exp(theta * t)  # solution to dh/dt = theta*h, h(0)=1

ax2.plot(t, h, color='steelblue', lw=2.5, label='$h(t) = e^{\\theta t}$')

# Mark a few points on the curve
sample_t = [0, 0.25, 0.5, 0.75, 1.0]
sample_h = np.exp(theta * np.array(sample_t))
ax2.scatter(sample_t, sample_h, color='steelblue', s=60, zorder=5)

ax2.scatter([0], [1], color='green', s=100, zorder=6, label='$h(0) = x$')
ax2.scatter([1], [np.exp(theta)], color='crimson', s=100, zorder=6, label='$h(T)$ = output')

ax2.set_xlabel('$t$')
ax2.set_ylabel('$h(t)$')
ax2.legend()
ax2.text(0.5, 0.15, 'ODE solver — infinite "layers", O(1) memory',
         ha='center', transform=ax2.transAxes, fontsize=10, color='#555', style='italic')

plt.tight_layout()
plt.suptitle('ResNet $\\to$ Neural ODE: letting layers become continuous', 
             y=1.02, fontsize=13)
plt.show()

---
<a id='setup'></a>
## 2. Setup: a simple scalar ODE

We use the simplest possible ODE so the algebra stays transparent:

$$\frac{dh}{dt} = \theta \cdot h(t), \qquad h(0) = 1, \quad t \in [0, 1]$$

This has the analytic solution $h(t) = e^{\theta t}$.

The **loss** is the squared error at $T=1$ against a target $y$:

$$L = \frac{1}{2}(h(1) - y)^2$$

We want: $\dfrac{dL}{d\theta}$

In [ ]:
theta_true = 2.0   # the 'true' parameter we want to recover
y_target = np.exp(theta_true)  # h(1) under theta_true = e^2

thetas = np.linspace(0, 4, 200)
h_at_T = np.exp(thetas)         # h(1) = e^theta
losses = 0.5 * (h_at_T - y_target)**2

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: solution curves for different theta
ax = axes[0]
t = np.linspace(0, 1, 200)
for th, col, lbl in [(0.5, '#aac8e8', '$\\theta=0.5$'),
                      (1.5, '#5b9bd5', '$\\theta=1.5$'),
                      (2.0, '#1a5fa8', '$\\theta=2.0$ (true)'),
                      (3.0, '#0d3060', '$\\theta=3.0$')]:
    ax.plot(t, np.exp(th*t), color=col, lw=2 + (th==2.0), 
            label=lbl, ls='-' if th==2.0 else '--')

ax.axhline(y=y_target, color='crimson', lw=1.5, ls=':', label=f'target $y={y_target:.2f}$')
ax.set_xlabel('$t$'); ax.set_ylabel('$h(t)$')
ax.set_title('Solution $h(t)=e^{\\theta t}$ for various $\\theta$')
ax.legend(fontsize=9)

# Right: loss landscape
ax2 = axes[1]
ax2.plot(thetas, losses, color='steelblue', lw=2)
ax2.axvline(x=theta_true, color='crimson', lw=1.5, ls='--', label=f'minimum at $\\theta={theta_true}$')
ax2.set_xlabel('$\\theta$'); ax2.set_ylabel('$L(\\theta)$')
ax2.set_title('Loss landscape $L = \\frac{1}{2}(e^\\theta - y)^2$')
ax2.legend()

plt.tight_layout()
plt.show()

print(f"Target y = e^{theta_true} = {y_target:.4f}")
print(f"Loss minimum at theta = {theta_true} (L = 0)")

---
<a id='naive'></a>
## 3. The naive gradient approach

### By direct differentiation (works for our simple case)

$$\frac{dL}{d\theta} = (h(1) - y) \cdot \frac{d}{d\theta} e^\theta = (e^\theta - y) \cdot e^\theta$$

### The problem with numerical solvers

For a general $f_\theta$, the ODE solver takes $N$ discrete steps. Naive backprop would:

1. **Store all $N$ intermediate states** $h(t_0), h(t_1), \ldots, h(t_N)$ in memory
2. Backpropagate through each solver step in reverse

Memory cost: $O(N)$ — this becomes prohibitive for stiff ODEs requiring many steps, or high-dimensional systems.

In [ ]:
def true_gradient(theta, y):
    """Analytic gradient dL/dtheta for our simple ODE."""
    h_T = np.exp(theta)
    return (h_T - y) * h_T  # chain rule: dL/dh(T) * dh(T)/dtheta

theta_test = 1.0
grad = true_gradient(theta_test, y_target)
print(f"At theta={theta_test}: analytic gradient = {grad:.4f}")
print(f"  dL/dh(T) = {np.exp(theta_test) - y_target:.4f}")
print(f"  dh(T)/dtheta = e^theta = {np.exp(theta_test):.4f}")

# Visualise gradient as tangent to loss curve
fig, ax = plt.subplots(figsize=(8, 4))

thetas_plot = np.linspace(0, 4, 200)
losses_plot = 0.5 * (np.exp(thetas_plot) - y_target)**2
ax.plot(thetas_plot, losses_plot, color='steelblue', lw=2, label='$L(\\theta)$')

# Tangent line at theta_test
L_at_test = 0.5 * (np.exp(theta_test) - y_target)**2
tang_theta = np.linspace(theta_test - 0.5, theta_test + 0.5, 50)
tang_L = L_at_test + grad * (tang_theta - theta_test)
ax.plot(tang_theta, tang_L, 'r--', lw=2, label=f'Tangent (gradient={grad:.2f})')
ax.scatter([theta_test], [L_at_test], color='red', s=80, zorder=5)
ax.axvline(x=theta_true, color='green', lw=1.5, ls=':', alpha=0.7, label='True minimum')

ax.set_xlabel('$\\theta$'); ax.set_ylabel('$L$')
ax.set_title(f'Gradient at $\\theta={theta_test}$: slope = {grad:.2f}')
ax.legend(); ax.set_ylim(-2, 20)
plt.tight_layout()
plt.show()

---
<a id='adjoint'></a>
## 4. Deriving the adjoint equation

The adjoint method treats the ODE as a **constraint** and uses Lagrange multipliers.

### Step 1: Augment the loss

Introduce adjoint variable $a(t)$ (a Lagrange multiplier function). Since the ODE residual is zero, adding it doesn't change $L$:

$$\tilde{L} = L - \int_0^T a(t) \underbrace{\left[\frac{dh}{dt} - f_\theta(h,t)\right]}_{=0} dt = L$$

### Step 2: Integrate by parts

The $\int a \dot{h}$ term is awkward. Integrate by parts:

$$\int_0^T a(t)\frac{dh}{dt}dt = \Big[a(t)h(t)\Big]_0^T - \int_0^T \dot{a}(t) h(t)\, dt$$

Substituting back:

$$\tilde{L} = L - \Big[a(T)h(T) - a(0)h(0)\Big] + \int_0^T \left[\dot{a}(t) + a(t)\theta\right] h(t)\, dt$$

### Step 3: Choose $a(t)$ to kill the integral

We **choose** $a(t)$ to satisfy:

$$\boxed{\frac{da}{dt} = -a(t)\theta}$$

This is the **adjoint ODE**. With terminal condition $a(T) = \frac{dL}{dh(T)} = h(T) - y$.

Its solution (for our example): $a(t) = a(T)\cdot e^{-\theta(t-T)}$

### Step 4: Read off $dL/d\theta$

With the integral killed, differentiating $\tilde{L}$ with respect to $\theta$ gives:

$$\frac{dL}{d\theta} = \int_0^T a(t) \cdot \frac{\partial f}{\partial \theta}\, dt = \int_0^T a(t) \cdot h(t)\, dt$$

(using $\partial f/\partial\theta = h$ for $f = \theta h$)

In [ ]:
def forward_solution(theta, t):
    """h(t) = exp(theta * t)"""
    return np.exp(theta * t)

def adjoint_solution(theta, y, t):
    """a(t) = a(T) * exp(-theta*(t - T)), T=1"""
    T = 1.0
    h_T = np.exp(theta * T)
    a_T = h_T - y          # terminal condition: dL/dh(T)
    return a_T * np.exp(-theta * (t - T))

theta_demo = 1.5
t = np.linspace(0, 1, 300)
h = forward_solution(theta_demo, t)
a = adjoint_solution(theta_demo, y_target, t)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Panel 1: Forward solution
axes[0].plot(t, h, color='steelblue', lw=2.5)
axes[0].scatter([0], [1], color='green', s=100, zorder=5, label='$h(0)=1$')
axes[0].scatter([1], [np.exp(theta_demo)], color='crimson', s=100, zorder=5,
                label=f'$h(1)={np.exp(theta_demo):.2f}$')
axes[0].set_title('Forward: $h(t) = e^{\\theta t}$', fontweight='bold')
axes[0].set_xlabel('$t$'); axes[0].set_ylabel('$h(t)$')
axes[0].legend(); axes[0].set_xlim(0, 1)

# Panel 2: Adjoint solution (runs backward)
axes[1].plot(t, a, color='darkorange', lw=2.5)
h_T = np.exp(theta_demo)
a_T = h_T - y_target
axes[1].scatter([1], [a_T], color='crimson', s=100, zorder=5,
                label=f'$a(T)=h(T)-y={a_T:.2f}$')
axes[1].annotate('Initialized here,\nsolves backward →', xy=(1, a_T),
                  xytext=(0.6, a_T+0.3),
                  arrowprops=dict(arrowstyle='->', color='gray'),
                  fontsize=9, color='gray')
axes[1].set_title('Adjoint: $a(t) = a(T)e^{-\\theta(t-T)}$', fontweight='bold')
axes[1].set_xlabel('$t$'); axes[1].set_ylabel('$a(t)$')
axes[1].legend(fontsize=9); axes[1].set_xlim(0, 1)

# Panel 3: Integrand a(t)*h(t), gradient is area under this curve
integrand = a * h
gradient_adjoint = np.trapezoid(integrand, t)
gradient_analytic = true_gradient(theta_demo, y_target)

axes[2].fill_between(t, 0, integrand, alpha=0.3, color='purple')
axes[2].plot(t, integrand, color='purple', lw=2.5)
axes[2].axhline(0, color='gray', lw=0.8)
axes[2].set_title('Integrand $a(t) \\cdot h(t)$', fontweight='bold')
axes[2].set_xlabel('$t$'); axes[2].set_ylabel('$a(t)\\cdot h(t)$')
axes[2].text(0.5, 0.85, f'Area = $dL/d\\theta$ = {gradient_adjoint:.4f}',
             transform=axes[2].transAxes, ha='center',
             bbox=dict(boxstyle='round', facecolor='lavender', alpha=0.8))
axes[2].set_xlim(0, 1)

plt.suptitle(f'Adjoint method at $\\theta={theta_demo}$', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

print(f"Adjoint gradient:  {gradient_adjoint:.6f}")
print(f"Analytic gradient: {gradient_analytic:.6f}")
print(f"Relative error:    {abs(gradient_adjoint - gradient_analytic)/abs(gradient_analytic)*100:.4f}%")

---
<a id='visualise'></a>
## 5. Visualising the adjoint: sensitivity over time

The adjoint $a(t)$ has a clear interpretation:

> **$a(t)$ measures how sensitive the loss $L$ is to a perturbation of $h(t)$ at time $t$.**

It's the gradient of $L$ with respect to the state at each moment in time, flowing *backward* from the terminal condition.

- At $t=T$: $a(T) = dL/dh(T)$ — direct gradient from the loss.
- At $t < T$: $a(t)$ propagates this sensitivity backward through the dynamics.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

thetas_vis = [0.5, 1.5, 2.5]

for col, th in enumerate(thetas_vis):
    h_t = forward_solution(th, t)
    a_t = adjoint_solution(th, y_target, t)
    integrand = a_t * h_t
    grad_adj = np.trapezoid(integrand, t)
    grad_true = true_gradient(th, y_target)
    
    # Top row: h(t) and a(t) together
    ax_top = axes[0, col]
    ax_top.plot(t, h_t, color='steelblue', lw=2, label='$h(t)$ forward')
    ax_top2 = ax_top.twinx()
    ax_top2.plot(t, a_t, color='darkorange', lw=2, ls='--', label='$a(t)$ adjoint')
    ax_top2.set_ylabel('$a(t)$', color='darkorange')
    ax_top2.tick_params(axis='y', colors='darkorange')
    ax_top.set_title(f'$\\theta = {th}$\n(grad={grad_adj:.2f})', fontweight='bold')
    ax_top.set_xlabel('$t$')
    ax_top.set_ylabel('$h(t)$', color='steelblue')
    ax_top.tick_params(axis='y', colors='steelblue')
    lines1, labels1 = ax_top.get_legend_handles_labels()
    lines2, labels2 = ax_top2.get_legend_handles_labels()
    ax_top.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='upper left')
    
    # Bottom row: integrand with shaded area = gradient
    ax_bot = axes[1, col]
    color_fill = 'green' if grad_adj > 0 else 'red'
    ax_bot.fill_between(t, 0, integrand, alpha=0.35, color=color_fill)
    ax_bot.plot(t, integrand, color=color_fill, lw=2)
    ax_bot.axhline(0, color='gray', lw=0.8)
    ax_bot.set_xlabel('$t$')
    ax_bot.set_ylabel('$a(t)h(t)$')
    ax_bot.set_title(f'Integrand (area = $dL/d\\theta$ = {grad_adj:.3f})')
    sign_str = 'gradient > 0: increase θ raises L' if grad_adj > 0 else 'gradient < 0: increase θ lowers L'
    ax_bot.text(0.5, 0.05, sign_str, transform=ax_bot.transAxes,
               ha='center', fontsize=8, color=color_fill,
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.suptitle('Adjoint variables and gradients for three values of $\\theta$\n(target: $\\theta^* = 2.0$)',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
<a id='algorithm'></a>
## 6. The full adjoint algorithm

For a **general** neural ODE $\dot{h} = f_\theta(h, t)$ the same derivation gives:

| Step | What | Equation |
|------|------|----------|
| **Forward pass** | Solve forward ODE | $\dot{h} = f_\theta(h,t)$, $h(0)=x$ |
| **Terminal condition** | Gradient at $T$ | $a(T) = \partial L/\partial h(T)$ |
| **Adjoint ODE** | Solve backward | $\dot{a} = -a^\top \frac{\partial f}{\partial h}$ |
| **Parameter gradient** | Accumulate | $dL/d\theta = \int_T^0 a^\top \frac{\partial f}{\partial \theta}\, dt$ |

**Key:** Steps 3 & 4 are solved together in a single backward ODE solve.

In `torchdiffeq`, this is `odeint_adjoint` — it runs both the adjoint ODE and the parameter gradient accumulation simultaneously, using **O(1) memory** regardless of solver step count.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.axis('off')

# Draw a diagram of the algorithm
box_style = dict(boxstyle='round,pad=0.5', facecolor='#deeaf7', edgecolor='steelblue', linewidth=1.5)
box_style2 = dict(boxstyle='round,pad=0.5', facecolor='#e8d5f5', edgecolor='purple', linewidth=1.5)
box_style3 = dict(boxstyle='round,pad=0.5', facecolor='#fce8d0', edgecolor='darkorange', linewidth=1.5)

steps = [
    (0.12, 0.60, 'STEP 1\nForward ODE\n$\\dot{h} = f_\\theta(h,t)$\n$t: 0 \\to T$', box_style, 'steelblue'),
    (0.38, 0.60, 'STEP 2\nCompute loss\n$L = \\ell(h(T))$\n$a(T) = \\partial L/\\partial h(T)$', box_style, 'steelblue'),
    (0.65, 0.60, 'STEP 3\nAdjoint ODE (backward)\n$\\dot{a} = -a^T \\partial_h f$\n$t: T \\to 0$', box_style2, 'purple'),
    (0.65, 0.15, 'STEP 4\nAccumulate $dL/d\\theta$\n$\\int a^T \\partial_\\theta f\\, dt$\n(alongside Step 3)', box_style2, 'purple'),
    (0.38, 0.15, 'STEP 5\nUpdate $\\theta$\n$\\theta \\leftarrow \\theta - \\eta\\, dL/d\\theta$', box_style3, 'darkorange'),
]

for x, y, txt, style, col in steps:
    ax.text(x, y, txt, transform=ax.transAxes, ha='center', va='center',
            bbox=style, fontsize=9.5, color='#222',
            multialignment='center')

# Arrows
arrow_kw = dict(transform=ax.transAxes, ha='center', va='center',
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))
ax.annotate('', xy=(0.29, 0.60), xytext=(0.21, 0.60), **arrow_kw)
ax.annotate('', xy=(0.56, 0.60), xytext=(0.48, 0.60), **arrow_kw)
ax.annotate('', xy=(0.65, 0.38), xytext=(0.65, 0.46), **arrow_kw)
ax.annotate('', xy=(0.50, 0.15), xytext=(0.56, 0.15), **arrow_kw)
ax.annotate('', xy=(0.20, 0.15), xytext=(0.29, 0.15), **arrow_kw)

# Memory labels
ax.text(0.25, 0.78, 'stores only $h(T)$ \u2014 O(1)',
        transform=ax.transAxes, ha='center', fontsize=8.5,
        color='steelblue', style='italic')
ax.text(0.65, 0.82, 'backward pass: O(1) memory',
        transform=ax.transAxes, ha='center', fontsize=8.5,
        color='purple', style='italic')

ax.set_title('The Adjoint Algorithm for Neural ODEs', fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

---
<a id='demo'></a>
## 7. Numerical demo: gradient check

We verify the adjoint gradient against:
1. The analytic formula
2. Finite differences (numerical gradient)

across a range of $\theta$ values.

In [ ]:
def adjoint_gradient(theta, y, n_steps=1000):
    """Compute dL/dtheta via the adjoint method (numerical integration)."""
    T = 1.0
    t_grid = np.linspace(0, T, n_steps)
    h_t = forward_solution(theta, t_grid)
    a_t = adjoint_solution(theta, y, t_grid)
    # dL/dtheta = integral of a(t) * df/dtheta = a(t) * h(t)
    integrand = a_t * h_t
    return np.trapezoid(integrand, t_grid)

def finite_diff_gradient(theta, y, eps=1e-5):
    """Numerical gradient via finite differences."""
    h_plus  = np.exp(theta + eps)
    h_minus = np.exp(theta - eps)
    L_plus  = 0.5 * (h_plus  - y)**2
    L_minus = 0.5 * (h_minus - y)**2
    return (L_plus - L_minus) / (2*eps)

thetas_check = np.linspace(0.2, 3.8, 30)
grads_adjoint  = [adjoint_gradient(th, y_target) for th in thetas_check]
grads_analytic = [true_gradient(th, y_target)    for th in thetas_check]
grads_fd       = [finite_diff_gradient(th, y_target) for th in thetas_check]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: all three gradients
ax = axes[0]
ax.plot(thetas_check, grads_analytic, 'k-', lw=3, label='Analytic', zorder=3)
ax.plot(thetas_check, grads_adjoint, 'o', color='purple', ms=7, label='Adjoint method', zorder=4)
ax.plot(thetas_check, grads_fd, 's', color='darkorange', ms=5, alpha=0.8,
        label='Finite differences', zorder=2)
ax.axhline(0, color='gray', lw=0.8)
ax.axvline(x=theta_true, color='green', lw=1.5, ls=':', label='$\\theta^*$ (true)')
ax.set_xlabel('$\\theta$')
ax.set_ylabel('$dL/d\\theta$')
ax.set_title('Gradient comparison: all three methods agree')
ax.legend(fontsize=9)

# Right: relative errors
ax2 = axes[1]
analytic_arr = np.array(grads_analytic)
err_adj = np.abs(np.array(grads_adjoint) - analytic_arr) / (np.abs(analytic_arr) + 1e-8)
err_fd  = np.abs(np.array(grads_fd)      - analytic_arr) / (np.abs(analytic_arr) + 1e-8)

ax2.semilogy(thetas_check, err_adj + 1e-10, color='purple', lw=2, label='Adjoint method')
ax2.semilogy(thetas_check, err_fd  + 1e-10, color='darkorange', lw=2, ls='--', label='Finite differences')
ax2.set_xlabel('$\\theta$')
ax2.set_ylabel('Relative error (log scale)')
ax2.set_title('Relative error vs analytic gradient')
ax2.legend()
ax2.set_ylim(1e-10, 1e-2)

plt.tight_layout()
plt.show()

print(f"Mean relative error — adjoint:           {np.mean(err_adj):.2e}")
print(f"Mean relative error — finite differences: {np.mean(err_fd):.2e}")

---
<a id='memory'></a>
## 8. Why memory cost is O(1): the key insight

Standard backprop through an ODE solver with $N$ steps requires storing **all $N$ intermediate states**.

The adjoint method instead:
1. Runs the forward solve, saves **only $h(T)$**
2. Runs a **new** backward ODE solve for $a(t)$, recomputing $h(t)$ as needed on the fly

Memory cost grows with the **state dimension**, not the number of solver steps.

Below we show how naive backprop memory grows with $N$, while the adjoint stays flat.

In [ ]:
state_dim = 128  # dimension of h (e.g. hidden units in neural ODE)
bytes_per_float = 4  # float32

solver_steps = np.array([10, 50, 100, 200, 500, 1000, 2000, 5000])

# Naive backprop: store all N states
memory_naive = solver_steps * state_dim * bytes_per_float / 1e6  # MB

# Adjoint: only current state + adjoint (constant)
memory_adjoint = np.ones_like(solver_steps) * 2 * state_dim * bytes_per_float / 1e6

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(solver_steps, memory_naive, 'o-', color='crimson', lw=2, label=f'Naive backprop: O(N)')
ax.axhline(memory_adjoint[0], color='steelblue', lw=2.5, ls='--',
           label=f'Adjoint method: O(1) = {memory_adjoint[0]*1000:.1f} KB')
ax.set_xlabel('Number of solver steps $N$')
ax.set_ylabel('Memory (MB)')
ax.set_title(f'Memory cost (state dim = {state_dim})')
ax.legend()

# Right: show the 'price' — 2x compute
ax2 = axes[1]
compute_naive   = solver_steps  # 1x forward
compute_adjoint = 2 * solver_steps  # ~2x (forward + backward adjoint solve)

ax2.plot(solver_steps, compute_naive, 'o-', color='steelblue', lw=2, label='Forward pass')
ax2.plot(solver_steps, compute_adjoint, 's-', color='purple', lw=2, label='Adjoint total (~2x)')
ax2.set_xlabel('Number of solver steps $N$')
ax2.set_ylabel('Relative compute cost')
ax2.set_title('Compute cost: adjoint ~2x forward')
ax2.legend()

plt.suptitle('Adjoint method tradeoff: O(1) memory, ~2x compute', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("Memory comparison at N=1000 steps:")
print(f"  Naive backprop: {memory_naive[-2]:.3f} MB")
print(f"  Adjoint method: {memory_adjoint[0]*1000:.2f} KB")
print(f"  Ratio: {memory_naive[-2]/(memory_adjoint[0]):.0f}x less memory with adjoint")

---
## Summary

The adjoint method for neural ODEs works in four steps:

1. **Solve the forward ODE** from $t=0$ to $T$; keep only $h(T)$.
2. **Initialise the adjoint** at $t=T$: $a(T) = \partial L/\partial h(T)$.
3. **Solve the adjoint ODE backward** from $T$ to $0$: $\dot{a} = -a^\top \partial_h f$.
4. **Accumulate the parameter gradient** alongside step 3: $dL/d\theta = \int_T^0 a^\top \partial_\theta f\, dt$.

**Why it matters:**
- Memory: $O(1)$ in solver steps (vs $O(N)$ for naive backprop)
- Compute: only ~$2\times$ the forward pass cost
- Enables training of deep continuous-time models that would be impractical with standard backprop

**References:**
- Chen et al. (2018) — *Neural Ordinary Differential Equations*. NeurIPS. [arXiv:1806.07366](https://arxiv.org/abs/1806.07366)
- Pontryagin et al. (1962) — *Mathematical Theory of Optimal Processes* (original adjoint/costate formulation)
- `torchdiffeq` library: https://github.com/rtqichen/torchdiffeq